# Pandas III — Análisis avanzado con dataset real (Supermarket Sales)

## Objetivos
- Preparar datos reales para análisis (tipos, fechas, validaciones).
- Dominar `groupby` avanzado (`agg`, `transform`, rankings por grupo).
- Trabajar con series temporales (`resample`, `rolling`, `pct_change`).
- Construir pivots de negocio y tablas listas para reporting.
- Entender cuándo usar `.apply()` y cuándo evitarlo.

## Dataset
Usaremos el CSV **Supermarket Sales**. Si en algún momento necesitamos más complejidad (por ejemplo, identificador de cliente), generaremos columnas adicionales a partir del dataset (sin depender de datos aleatorios puros).

---
## 0. Setup

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
pd.set_option('display.max_rows', 60)

np.random.seed(42)
print('pandas:', pd.__version__)

pandas: 3.0.0


---
## 1. Carga del dataset y primera inspección

### Teoría
En proyectos reales, la primera pasada responde a:
- ¿Qué columnas hay?
- ¿Cuántas filas?
- ¿Qué tipos inferidos tiene?
- ¿Hay nulos?

Esto define el plan de limpieza y análisis.

### Ejemplo (carga + overview)

In [2]:
path = 'supermarket_sales - Sheet1.csv'
df = pd.read_csv(path)

print('shape:', df.shape)
display(df.head())
display(df.dtypes)
display(df.isna().sum().sort_values(ascending=False).head(10))

shape: (1000, 17)


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


Invoice ID                     str
Branch                         str
City                           str
Customer type                  str
Gender                         str
Product line                   str
Unit price                 float64
Quantity                     int64
Tax 5%                     float64
Total                      float64
Date                           str
Time                           str
Payment                        str
cogs                       float64
gross margin percentage    float64
gross income               float64
Rating                     float64
dtype: object

Invoice ID       0
Branch           0
City             0
Customer type    0
Gender           0
Product line     0
Unit price       0
Quantity         0
Tax 5%           0
Total            0
dtype: int64

### Ejercicio 1 (inspección)

1) Usa `df.info()`.
2) Calcula el número de valores únicos de:
- `Invoice ID`
- `City`
- `Product line`
- `Payment`
3) ¿Hay duplicados exactos? (`df.duplicated().sum()`).

Entrega: imprime resultados de forma clara.

In [3]:
df.info()

display(df[['Invoice ID', 'City', 'Product line', 'Payment']].nunique())

print('Exact duplicated rows:', df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   str    
 1   Branch                   1000 non-null   str    
 2   City                     1000 non-null   str    
 3   Customer type            1000 non-null   str    
 4   Gender                   1000 non-null   str    
 5   Product line             1000 non-null   str    
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Total                    1000 non-null   float64
 10  Date                     1000 non-null   str    
 11  Time                     1000 non-null   str    
 12  Payment                  1000 non-null   str    
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  1000 non-nu

Invoice ID      1000
City               3
Product line       6
Payment            3
dtype: int64

Exact duplicated rows: 0


---
## 2. Tipos, datetime y columnas derivadas

### Teoría
Para análisis temporal y reporting:
- `Date` debe ser datetime.
- `Time` puede integrarse con `Date` para crear `datetime`.
- Dimensiones con baja cardinalidad suelen ser candidatas a `category`.

Esto mejora consistencia y rendimiento.

### Ejemplo (parseo de fechas + datetime + features temporales)

In [4]:
df2 = df.copy()

# Parse Date
df2['Date'] = pd.to_datetime(df2['Date'])

# df2['Date'] = df2['Date'].apply(lambda x: pd.to_datetime(x)if isinstance(x, str) else np.nan)
display(df2['Date'].sample(5))

# Create datetime from Date + Time
df2['datetime'] = pd.to_datetime(df2['Date'].dt.strftime('%Y-%m-%d') + ' ' + df2['Time'])
display(df2[['Date','Time','datetime']].sample(5))


# Temporal features
df2['month'] = df2['datetime'].dt.to_period('M').astype(str)
df2['weekday'] = df2['datetime'].dt.day_name()
df2['hour'] = df2['datetime'].dt.hour

display(df2[['Date','Time','datetime','month','weekday','hour']].head())
display(df2.dtypes[['Date','datetime']])

521   2019-03-20
737   2019-01-29
740   2019-03-23
660   2019-02-03
411   2019-01-25
Name: Date, dtype: datetime64[us]

,Date,Time,datetime
93,2019-03-12,12:09,2019-03-12 12:09:00
692,2019-03-04,12:44,2019-03-04 12:44:00
100,2019-03-26,19:20,2019-03-26 19:20:00
562,2019-01-30,13:58,2019-01-30 13:58:00
729,2019-03-09,10:54,2019-03-09 10:54:00


,Date,Time,datetime,month,weekday,hour
0,2019-01-05,13:08,2019-01-05 13:08:00,2019-01,Saturday,13
1,2019-03-08,10:29,2019-03-08 10:29:00,2019-03,Friday,10
2,2019-03-03,13:23,2019-03-03 13:23:00,2019-03,Sunday,13
3,2019-01-27,20:33,2019-01-27 20:33:00,2019-01,Sunday,20
4,2019-02-08,10:37,2019-02-08 10:37:00,2019-02,Friday,10


Date        datetime64[us]
datetime    datetime64[us]
dtype: object

### Ejercicio 2 (tipos y categóricas)

1) Convierte a tipo `category` estas columnas:
- `Branch`, `City`, `Customer type`, `Gender`, `Product line`, `Payment`
2) Mide memoria antes y después (`memory_usage(deep=True).sum()`).

Entrega: imprime el ahorro de memoria.

In [5]:
# Memory before
mem_before = df2.memory_usage(deep=True).sum()

# Convert categoricals
cat_cols = ['Branch', 'City', 'Customer type', 'Gender', 'Product line', 'Payment']
for c in cat_cols:
    df2[c] = df2[c].astype('category')

# Memory after
mem_after = df2.memory_usage(deep=True).sum()
print('Memory before:', mem_before)
print('Memory after :', mem_after)
print('Saved (%)    :', round((mem_before - mem_after) / mem_before * 100, 2))

df2[['Date','Time','datetime','month','weekday','hour']].head()
display(df2.dtypes)

Memory before: 649725
Memory after : 317435
Saved (%)    : 51.14


Invoice ID                            str
Branch                           category
City                             category
Customer type                    category
Gender                           category
Product line                     category
Unit price                        float64
Quantity                            int64
Tax 5%                            float64
Total                             float64
Date                       datetime64[us]
Time                                  str
Payment                          category
cogs                              float64
gross margin percentage           float64
gross income                      float64
Rating                            float64
datetime                   datetime64[us]
month                                 str
weekday                               str
hour                                int32
dtype: object

---
## 3. Sanity checks (coherencia contable)

### Teoría
En datasets financieros, es obligatorio validar identidades:
- `Total ≈ cogs + Tax 5%`
- En este dataset: `gross income` suele coincidir con `Tax 5%`

Importante: usar tolerancia por redondeos.

In [6]:
0.1+0.2==0.3

False

### Ejemplo (comprobación con tolerancia)

In [7]:
tmp = df2.copy()
tol = 1e-6 # Tolerance for floating point comparison 1e-6 = 0.000001

check_total = np.isclose(tmp['Total'], tmp['cogs'] + tmp['Tax 5%'], atol=tol)
check_income = np.isclose(tmp['gross income'], tmp['Tax 5%'], atol=tol)

print('Total == cogs + Tax 5%:', check_total.mean())
print('gross income == Tax 5%:', check_income.mean())

display(tmp.loc[~check_total, ['cogs','Tax 5%','Total']].head())

Total == cogs + Tax 5%: 1.0
gross income == Tax 5%: 1.0


,cogs,Tax 5%,Total


### Ejercicio 3 (debug de inconsistencias)

1) Encuentra filas donde `Total` no sea consistente con `cogs + Tax 5%` (con tolerancia).
2) Calcula cuántas son y qué porcentaje representan.
3) Crea una columna `total_diff = Total - (cogs + Tax 5%)` y resume con `describe()`.

Entrega: tabla con las 10 mayores desviaciones absolutas.

In [8]:
tol = 1e-6
check_total = np.isclose(df2['Total'], df2['cogs'] + df2['Tax 5%'], atol=tol)
check_income = np.isclose(df2['gross income'], df2['Tax 5%'], atol=tol)

print('Total == cogs + Tax 5% ratio:', check_total.mean())
print('gross income == Tax 5% ratio:', check_income.mean())

bad = df2.loc[~check_total].copy()
bad['total_diff'] = bad['Total'] - (bad['cogs'] + bad['Tax 5%'])

print('Bad rows:', bad.shape[0], '(', round(bad.shape[0]/df2.shape[0]*100, 2), '% )')
display(bad['total_diff'].describe())

bad['abs_diff'] = bad['total_diff'].abs()
bad_sorted = bad.sort_values('abs_diff', ascending=False)
bad_sorted[['Invoice ID','cogs','Tax 5%','Total','total_diff','abs_diff']].head(10)

Total == cogs + Tax 5% ratio: 1.0
gross income == Tax 5% ratio: 1.0
Bad rows: 0 ( 0.0 % )


count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: total_diff, dtype: float64

,Invoice ID,cogs,Tax 5%,Total,total_diff,abs_diff


---
## 4. `groupby` avanzado: `agg` con múltiples métricas

### Teoría
Un buen resumen de negocio suele incluir:
- `total_sales` (sum de `Total`)
- `tx_count` (número de tickets)
- `avg_ticket` (media de `Total`)
- `avg_rating` (media de `Rating`)
- `total_gross_income` (sum de `gross income`)

La clave es nombrar columnas finales de forma legible.

### Ejemplo (ventas por ciudad)

In [9]:
display(df2.sample(5))

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,datetime,month,weekday,hour
612,789-23-8625,B,Mandalay,Member,Male,Fashion accessories,93.22,3,13.9830,293.6430,2019-01-24,11:45,Cash,279.66,4.761905,13.9830,7.2,2019-01-24 11:45:00,2019-01,Thursday,11
468,746-19-0921,C,Naypyitaw,Normal,Male,Food and beverages,21.58,1,1.0790,22.6590,2019-02-09,10:02,Ewallet,21.58,4.761905,1.0790,7.2,2019-02-09 10:02:00,2019-02,Saturday,10
355,889-04-9723,B,Mandalay,Member,Female,Food and beverages,89.14,4,17.8280,374.3880,2019-01-07,12:20,Credit card,356.56,4.761905,17.8280,7.8,2019-01-07 12:20:00,2019-01,Monday,12
586,278-86-2735,A,Yangon,Normal,Female,Food and beverages,52.34,3,7.8510,164.8710,2019-03-27,14:03,Cash,157.02,4.761905,7.8510,9.2,2019-03-27 14:03:00,2019-03,Wednesday,14
938,131-70-8179,A,Yangon,Member,Female,Health and beauty,92.09,3,13.8135,290.0835,2019-02-17,16:27,Cash,276.27,4.761905,13.8135,4.2,2019-02-17 16:27:00,2019-02,Sunday,16


In [10]:
city_kpis = df2.groupby('City', as_index=False).agg(
    total_sales=('Total', 'sum'),
    tx_count=('Invoice ID', 'count'),
    avg_ticket=('Total', 'mean'),
    total_gross_income=('gross income', 'sum'),
    avg_rating=('Rating', 'mean')
).sort_values('total_sales', ascending=False)

city_kpis

,City,total_sales,tx_count,avg_ticket,total_gross_income,avg_rating
1,Naypyitaw,110568.7065,328,337.099715,5265.1765,7.072866
2,Yangon,106200.3705,340,312.354031,5057.1605,7.027059
0,Mandalay,106197.6720,332,319.872506,5057.0320,6.818072


### Ejercicio 4 (KPIs por dimensión)

Crea tablas KPI (mismas métricas del ejemplo) por:
- `Branch`
- `Product line`
- `Payment`

Entrega:
- tres DataFrames ordenados por `total_sales` desc.
- muestra top 5 de cada uno.

In [11]:
def kpi_table(data: pd.DataFrame, dim: str) -> pd.DataFrame:
    out = data.groupby(dim, as_index=False).agg(
        total_sales=('Total', 'sum'),
        tx_count=('Invoice ID', 'count'),
        avg_ticket=('Total', 'mean'),
        total_gross_income=('gross income', 'sum'),
        avg_rating=('Rating', 'mean')
    )
    return out.sort_values('total_sales', ascending=False)

branch_kpis = kpi_table(df2, 'Branch')
line_kpis = kpi_table(df2, 'Product line')
payment_kpis = kpi_table(df2, 'Payment')

display(branch_kpis.head(2))
display(line_kpis.head(3))
display(payment_kpis.head(2))

,Branch,total_sales,tx_count,avg_ticket,total_gross_income,avg_rating
2,C,110568.7065,328,337.099715,5265.1765,7.072866
0,A,106200.3705,340,312.354031,5057.1605,7.027059


,Product line,total_sales,tx_count,avg_ticket,total_gross_income,avg_rating
2,Food and beverages,56144.8440,174,322.671517,2673.5640,7.113218
5,Sports and travel,55122.8265,166,332.065220,2624.8965,6.916265
0,Electronic accessories,54337.5315,170,319.632538,2587.5015,6.924706


,Payment,total_sales,tx_count,avg_ticket,total_gross_income,avg_rating
0,Cash,112206.570,344,326.18189,5343.170,6.970058
2,Ewallet,109993.107,345,318.82060,5237.767,6.947826


---
## 5. `transform`: métricas por fila basadas en grupo

### Teoría
`transform` se usa cuando quieres asignar a cada fila un valor calculado sobre su grupo.
Ejemplos:
- total mensual al que pertenece la fila
- share de una fila respecto a su ciudad
- z-score de `Total` dentro de `Product line`


### Ejemplo (share del ticket dentro de su ciudad)

In [12]:
fs = df2.copy()
fs['city_total_sales'] = fs.groupby('City')['Total'].transform('sum')
fs['ticket_share_city'] = fs['Total'] / fs['city_total_sales']
fs[['City','Total','city_total_sales','ticket_share_city']].head()

,City,Total,city_total_sales,ticket_share_city
0,Yangon,548.9715,106200.3705,0.005169
1,Naypyitaw,80.2200,110568.7065,0.000726
2,Yangon,340.5255,106200.3705,0.003206
3,Yangon,489.0480,106200.3705,0.004605
4,Yangon,634.3785,106200.3705,0.005973


### Ejercicio 5 (transform)

1) Crea `product_line_total_sales` (sum de `Total` por `Product line`) con `transform`.
2) Crea `ticket_share_product_line`.
3) Crea `z_total_in_line` (z-score de `Total` dentro de `Product line`).

Entrega: muestra 10 filas con esas columnas.

In [13]:
fs = df2.copy()

fs['product_line_total_sales'] = fs.groupby('Product line')['Total'].transform('sum')
fs['ticket_share_product_line'] = fs['Total'] / fs['product_line_total_sales']

# z-score within Product line
mean_line = fs.groupby('Product line')['Total'].transform('mean')
std_line = fs.groupby('Product line')['Total'].transform('std').replace(0, 1) # Avoid division by zero
fs['z_total_in_line'] = (fs['Total'] - mean_line) / std_line

fs[['Product line','Total','product_line_total_sales','ticket_share_product_line','z_total_in_line']].head(10)

,Product line,Total,product_line_total_sales,ticket_share_product_line,z_total_in_line
0,Health and beauty,548.9715,49193.7390,0.011159,0.948596
1,Electronic accessories,80.2200,54337.5315,0.001476,-0.973437
2,Home and lifestyle,340.5255,53861.9130,0.006322,0.015273
3,Health and beauty,489.0480,49193.7390,0.009941,0.696328
4,Sports and travel,634.3785,55122.8265,0.011508,1.217163
5,Electronic accessories,627.6165,54337.5315,0.011550,1.252244
6,Electronic accessories,433.6920,54337.5315,0.007981,0.463759
7,Home and lifestyle,772.3800,53861.9130,0.014340,1.711476
8,Health and beauty,76.1460,49193.7390,0.001548,-1.041922
9,Food and beverages,172.7460,56144.8440,0.003077,-0.606598


---
## 6. Top-N por grupo (rankings)

### Teoría
Patrones típicos:
- ventas por (grupo, item) → ordenar → top-N por grupo
- usar `nlargest`, o `rank` y filtrar

Esto aparece en dashboards continuamente: top productos por ciudad, top pagos por sucursal, etc.

In [14]:

df_test1 = pd.DataFrame({
    'Producto': ['A', 'B', 'C', 'D', 'E'],
    'Ventas': [100, 500, 300, 500, 200]
})

# Obtener los 3 productos con más ventas
top_3 = df_test1.nlargest(3, 'Ventas')
display(top_3)

,Producto,Ventas
1,B,500
3,D,500
2,C,300


In [15]:
# Crear la columna de ranking
# ascending=False: el valor más alto es el 1
df_test1['Posicion'] = df_test1['Ventas'].rank(ascending=False, method='min')

# Filtrar: Mostrar solo los que están en el Top 2 (incluyendo empates)
top_rank = df_test1[df_test1['Posicion'] <= 2]
display(df_test1)
display(top_rank)

,Producto,Ventas,Posicion
0,A,100,5.0
1,B,500,1.0
2,C,300,3.0
3,D,500,1.0
4,E,200,4.0


,Producto,Ventas,Posicion
1,B,500,1.0
3,D,500,1.0


### Ejemplo (top 3 líneas de producto por ciudad)

In [16]:
city_line_sales = (
    df2.groupby(['City','Product line'], as_index=False)
       .agg(total_sales=('Total','sum'))
)

top3_by_city = (
    city_line_sales.sort_values(['City','total_sales'], ascending=[True, False])
                 .groupby('City', as_index=False)
                 .head(3)
)

top3_by_city

,City,Product line,total_sales
5,Mandalay,Sports and travel,19988.1990
3,Mandalay,Health and beauty,19980.6600
4,Mandalay,Home and lifestyle,17549.1645
8,Naypyitaw,Food and beverages,23766.8550
7,Naypyitaw,Fashion accessories,21560.0700
6,Naypyitaw,Electronic accessories,18968.9745
16,Yangon,Home and lifestyle,22417.1955
17,Yangon,Sports and travel,19372.6995
12,Yangon,Electronic accessories,18317.1135


### Ejercicio 6 (rankings)

1) Top 2 métodos de pago (`Payment`) por `Branch` (por ventas).
2) Top 3 líneas de producto por `Branch` (por `gross income`).

Entrega:
- dos tablas con columnas de grupo + item + métrica.


In [17]:
# 1) Top 2 Payment by Branch (sales)
branch_payment = (
    df2.groupby(['Branch','Payment'], as_index=False)
       .agg(total_sales=('Total','sum'))
       .sort_values(['Branch','total_sales'], ascending=[True, False])
)
top2_payment_by_branch = branch_payment.groupby('Branch', as_index=False).head(2)

# 2) Top 3 Product line by Branch (gross income)
branch_line_profit = (
    df2.groupby(['Branch','Product line'], as_index=False)
       .agg(total_gross_income=('gross income','sum'))
       .sort_values(['Branch','total_gross_income'], ascending=[True, False])
)
top3_line_by_branch_profit = branch_line_profit.groupby('Branch', as_index=False).head(3)

display(top2_payment_by_branch)
display(top3_line_by_branch_profit)

,Branch,Payment,total_sales
2,A,Ewallet,39324.3690
0,A,Cash,33781.2510
4,B,Credit card,37344.8565
3,B,Cash,35339.4615
6,C,Cash,43085.8575
8,C,Ewallet,37155.3840


,Branch,Product line,total_gross_income
4,A,Home and lifestyle,1067.4855
5,A,Sports and travel,922.5095
0,A,Electronic accessories,872.2435
11,B,Sports and travel,951.8190
9,B,Health and beauty,951.4600
10,B,Home and lifestyle,835.6745
14,C,Food and beverages,1131.7550
13,C,Fashion accessories,1026.6700
12,C,Electronic accessories,903.2845


---
## 7. `.apply()` (y su alternativa vectorizada)

### Teoría
`.apply()` es útil cuando:
- la lógica no es fácil de vectorizar
- necesitas una función clara y testeable

Pero suele ser más lento que:
- `pd.cut`
- `np.select`
- `map`/`replace`


### Ejemplo 1 (apply: ticket band por `Total`)

In [18]:
def ticket_band(x: float) -> str:
    if x < 100:
        return 'Low'
    if x < 300:
        return 'Medium'
    return 'High'

apply_demo = df2.copy()
apply_demo['ticket_band'] = apply_demo['Total'].apply(ticket_band)
apply_demo['ticket_band'].value_counts()


ticket_band
High      431
Medium    361
Low       208
Name: count, dtype: int64

### Ejemplo 2 (vectorizado: `pd.cut` para el mismo problema)

In [19]:
vec_demo = df2.copy()
bins = [-np.inf, 100, 300, np.inf]
labels = ['Low', 'Medium', 'High']
vec_demo['ticket_band'] = pd.cut(vec_demo['Total'], bins=bins, labels=labels)
vec_demo['ticket_band'].value_counts()

ticket_band
High      431
Medium    361
Low       208
Name: count, dtype: int64

### Ejercicio 7 (apply vs vectorizado)

1) Crea `rating_band` a partir de `Rating` con `apply`:
- `< 6` → Low
- `< 8` → Mid
- `>= 8` → High

2) Crea la misma columna con una alternativa vectorizada (elige: `pd.cut` o `np.select`).
3) Verifica que ambas columnas coinciden en al menos el 99% de filas.


In [20]:
# TODO
pass

---
## 8. Series temporales: resample, rolling, growth

### Teoría
Pasos típicos:
1) construir serie diaria (ventas por día)
2) calcular tendencia con media móvil (rolling)
3) calcular crecimiento (`pct_change`) y comparar periodos

### Medias Moviles
* De los ultimos X tiempos lo que quieres ver segun periodos


### Ejemplo (serie diaria + rolling 7 días)

In [21]:
ts = df2.set_index('datetime').sort_index()
daily_sales = ts['Total'].resample('D').sum()
daily_sales = daily_sales.fillna(0)

daily_roll7 = daily_sales.rolling(7, min_periods=1).mean()
daily_growth = daily_sales.pct_change().replace([np.inf, -np.inf], np.nan)

daily_df = pd.DataFrame({
    'daily_sales': daily_sales,
    'rolling_7d': daily_roll7,
    'pct_change': daily_growth
}).reset_index().rename(columns={'datetime': 'date'})

daily_df.head(10)

,date,daily_sales,rolling_7d,pct_change
0,2019-01-01,4745.1810,4745.18100,NaN
1,2019-01-02,1945.5030,3345.34200,-0.590004
2,2019-01-03,2078.1285,2922.93750,0.068170
3,2019-01-04,1623.6885,2598.12525,-0.218678
4,2019-01-05,3536.6835,2785.83690,1.178179
5,2019-01-06,3614.2050,2923.89825,0.021919
6,2019-01-07,2834.2440,2911.09050,-0.215804
7,2019-01-08,5293.7325,2989.45500,0.867776
8,2019-01-09,3021.3435,3143.14650,-0.429260
9,2019-01-10,3560.9490,3354.97800,0.178598


### Ejercicio 8 (time series)

1) Construye una serie semanal (`W`) de ventas (`Total`).
2) Calcula media móvil 4 semanas.
3) Devuelve un DataFrame con `week`, `weekly_sales`, `rolling_4w`.


In [22]:
# TODO
pass

### Ejercicio 9 (patrones por hora y día)

1) Ventas totales por `weekday`.
2) Ventas totales por `hour`.
3) Encuentra la combinación `weekday` + `hour` con mayor venta total.


In [23]:
df2.sample(5)

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,datetime,month,weekday,hour
555,356-44-8813,B,Mandalay,Normal,Male,Home and lifestyle,37.48,3,5.6220,118.0620,2019-01-20,13:45,Credit card,112.44,4.761905,5.6220,7.7,2019-01-20 13:45:00,2019-01,Sunday,13
395,726-27-2396,A,Yangon,Normal,Female,Health and beauty,77.50,5,19.3750,406.8750,2019-01-24,20:36,Ewallet,387.50,4.761905,19.3750,4.3,2019-01-24 20:36:00,2019-01,Thursday,20
527,339-38-9982,B,Mandalay,Member,Male,Fashion accessories,59.86,2,5.9860,125.7060,2019-01-13,14:55,Ewallet,119.72,4.761905,5.9860,6.7,2019-01-13 14:55:00,2019-01,Sunday,14
335,527-09-6272,A,Yangon,Member,Female,Electronic accessories,28.45,5,7.1125,149.3625,2019-03-21,10:17,Credit card,142.25,4.761905,7.1125,9.1,2019-03-21 10:17:00,2019-03,Thursday,10
666,779-42-2410,B,Mandalay,Member,Male,Food and beverages,57.74,3,8.6610,181.8810,2019-02-20,13:06,Ewallet,173.22,4.761905,8.6610,7.7,2019-02-20 13:06:00,2019-02,Wednesday,13


In [24]:
# Agrupar por el dia de la semana, y calcular la media de ventas por dia
# Como tenemos weekday lo podemos utilizar sino habria que poner: ts['Total'].resample('W-MON').sum().fillna(0) o similar dependiendo de la frecuencia
weekday_sales = df2.groupby('weekday', as_index=False).agg(
    total_sales=('Total', 'sum')
).sort_values('total_sales', ascending=False)

hourday_sales = df2.groupby('hour', as_index=False).agg(
    total_sales=('Total', 'sum')
).sort_values('total_sales', ascending=False)

# Por media cambia varias cosas, verlo y analizarlo
best_combo_sum = (
    df2.groupby(['weekday','hour'], as_index=False)
        .agg(total_sales=('Total', 'sum'))
        .sort_values('total_sales', ascending=False)
        .head(10)
)

best_combo_mean = (
    df2.groupby(['weekday','hour'], as_index=False)
        .agg(total_sales=('Total', 'mean'))
        .sort_values('total_sales', ascending=False)
        .head(10)
)

display(weekday_sales)
display(hourday_sales)
display(best_combo_sum)
display(best_combo_mean)

,weekday,total_sales
2,Saturday,56120.8095
5,Tuesday,51482.2455
4,Thursday,45349.2480
3,Sunday,44457.8925
0,Friday,43926.3405
6,Wednesday,43731.1350
1,Monday,37899.0780


,hour,total_sales
9,19,39699.5130
3,13,34723.2270
0,10,31421.4810
5,15,31179.5085
4,14,30828.3990
1,11,30377.3295
2,12,26065.8825
8,18,26030.3400
6,16,25226.3235
7,17,24445.2180


,weekday,hour,total_sales
64,Tuesday,19,9198.6720
31,Saturday,19,9117.3810
60,Tuesday,15,7020.2055
44,Thursday,10,6885.2280
3,Friday,13,6824.3070
30,Saturday,18,6738.7530
69,Wednesday,13,6734.3430
42,Sunday,19,5963.6430
25,Saturday,13,5689.6350
0,Friday,10,5671.6905


,weekday,hour,total_sales
60,Tuesday,15,501.443250
26,Saturday,14,485.625000
23,Saturday,11,457.242625
54,Thursday,20,452.523750
34,Sunday,11,430.615500
35,Sunday,12,429.836591
3,Friday,13,426.519187
61,Tuesday,16,425.715500
29,Saturday,17,422.484125
9,Friday,19,421.225269


---
## 9. Pivot tables (reporting-ready)

### Teoría
`pivot_table` convierte datos "long" en matrices útiles para reporting.
Luego es común:
- rellenar nulos
- añadir totales
- ordenar


### Ejemplo (City × Payment con totales)

In [25]:
pv = pd.pivot_table(
    df2, values='Total', index='City', columns='Payment', aggfunc='sum'
    ).fillna(0)
pv['Total'] = pv.sum(axis=1)
pv = pv.sort_values('Total', ascending=False)
pv.loc['Total'] = pv.sum() # Suma total por columna, iloc seria por fila
pv

Payment,Cash,Credit card,Ewallet,Total
City,,,,
Naypyitaw,43085.8575,30327.4650,37155.384,110568.7065
Yangon,33781.2510,33094.7505,39324.369,106200.3705
Mandalay,35339.4615,37344.8565,33513.354,106197.6720
Total,112206.5700,100767.0720,109993.107,322966.7490


### Ejercicio 10 (pivots)

Crea estos pivots:
1) `Branch` × `Product line` con suma de `Total`.
2) `City` × `Gender` con media de `Rating`.
3) `month` × `Product line` con suma de `gross income`.

Entrega:
- rellena nulos con 0 cuando aplique
- añade columna `Total` por fila en (1) y (3)


In [26]:
pv_branch_pLine = pd.pivot_table(
    df2, values = 'Total', index= 'Branch', columns='Product line', aggfunc='sum'
    ).fillna(0)     # observed=False para incluir combinaciones no observadas, aunque en este caso no hay porque Branch y Product line son categoricas sin orden
pv_branch_pLine['Total'] = pv_branch_pLine.sum(axis=1)
pv_branch_pLine = pv_branch_pLine.sort_values('Total', ascending=False)
display(pv_branch_pLine)

pv_city = pd.pivot_table(
    df2, values='Rating', index='City', columns='Gender', aggfunc='mean'
).fillna(0)
pv_city['Average'] = pv_city.mean(axis=1)
pv_city = pv_city.sort_values('Average', ascending=False)
display(pv_city)

pv_month = pd.pivot_table(
    df2, values='gross income', index='month', columns='Product line', aggfunc='sum' 
).fillna(0)
pv_month['Total'] = pv_month.sum(axis=1)
pv_month = pv_month.sort_values('Total', ascending=False)
display(pv_month)

Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel,Total
Branch,,,,,,,
C,18968.9745,21560.0700,23766.8550,16615.326,13895.5530,15761.9280,110568.7065
A,18317.1135,16332.5085,17163.1005,12597.753,22417.1955,19372.6995,106200.3705
B,17051.4435,16413.3165,15214.8885,19980.660,17549.1645,19988.1990,106197.6720


Gender,Female,Male,Average
City,,,
Naypyitaw,7.157865,6.972000,7.064933
Yangon,6.839130,7.196089,7.017610
Mandalay,6.876543,6.762353,6.819448


Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel,Total
month,,,,,,,
2019-01,896.7280,921.1960,931.930,780.1510,975.9400,1031.7630,5537.708
2019-03,863.9685,759.5675,789.236,867.0625,996.7995,935.5330,5212.167
2019-02,826.8050,905.2315,952.398,695.3455,592.1135,657.6005,4629.494


---
## 10. Enriquecimiento opcional: crear un "customer_id" (sin datos aleatorios puros)

### Teoría
Este dataset no trae identificador real de cliente. Para poder practicar análisis por cliente,
crearemos un pseudo-id determinista a partir de columnas existentes (y un pequeño componente controlado).

Idea:
- usar City, Branch, Customer type, Gender   (Datetime es para saber cuantos ha habido)
- y asignar un id estable por combinación + un contador

Esto no es "real" pero sirve para practicar técnicas de análisis (VIP, Pareto, etc.).

### Ejemplo (pseudo customer_id)

Creamos un id por grupo y lo repartimos de forma determinista dentro del grupo.

In [27]:
enriched = df2.copy().sort_values(['City','Branch','Customer type','Gender','datetime'])

# group signature
sig_cols = ['City','Branch','Customer type','Gender']
enriched['sig'] = enriched[sig_cols].astype(str).agg('|'.join, axis=1)

# deterministic customer assignment: cycle ids within each signature
customers_per_sig = 25
enriched['sig_rank'] = enriched.groupby('sig').cumcount() #Aqui que hace cumcount? Los va sumando
display(enriched[['sig','sig_rank']].head(20))
enriched['customer_id'] = enriched['sig'] + '::C' + (enriched['sig_rank'] % customers_per_sig).astype(str)
display(enriched[['sig','sig_rank','customer_id']].head(10))

enriched[['sig','customer_id']].head(10)

,sig,sig_rank
970,Mandalay|B|Member|Female,0
407,Mandalay|B|Member|Female,1
536,Mandalay|B|Member|Female,2
376,Mandalay|B|Member|Female,3
355,Mandalay|B|Member|Female,4
67,Mandalay|B|Member|Female,5
686,Mandalay|B|Member|Female,6
506,Mandalay|B|Member|Female,7
685,Mandalay|B|Member|Female,8
880,Mandalay|B|Member|Female,9


,sig,sig_rank,customer_id
970,Mandalay|B|Member|Female,0,Mandalay|B|Member|Female::C0
407,Mandalay|B|Member|Female,1,Mandalay|B|Member|Female::C1
536,Mandalay|B|Member|Female,2,Mandalay|B|Member|Female::C2
376,Mandalay|B|Member|Female,3,Mandalay|B|Member|Female::C3
355,Mandalay|B|Member|Female,4,Mandalay|B|Member|Female::C4
67,Mandalay|B|Member|Female,5,Mandalay|B|Member|Female::C5
686,Mandalay|B|Member|Female,6,Mandalay|B|Member|Female::C6
506,Mandalay|B|Member|Female,7,Mandalay|B|Member|Female::C7
685,Mandalay|B|Member|Female,8,Mandalay|B|Member|Female::C8
880,Mandalay|B|Member|Female,9,Mandalay|B|Member|Female::C9


,sig,customer_id
970,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C0
407,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C1
536,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C2
376,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C3
355,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C4
67,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C5
686,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C6
506,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C7
685,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C8
880,Mandalay|B|Member|Female,Mandalay|B|Member|Female::C9


### Ejercicio 11 (análisis por cliente)

Usando `enriched`:
1) Calcula gasto total por `customer_id`.
2) Define VIP como top 10% por gasto.
3) Calcula qué % de ventas total aportan los VIP.

Entrega:
- número de VIP
- ventas VIP vs no-VIP
- share VIP


In [28]:
enriched = df2.copy().sort_values(['City','Branch','Customer type','Gender','datetime'])

# group signature
sig_cols = ['City','Branch','Customer type','Gender']
enriched['sig'] = enriched[sig_cols].astype(str).agg('|'.join, axis=1)

customers_per_sig = 25
enriched['sig_rank'] = enriched.groupby('sig').cumcount()
enriched['customer_id'] = enriched['sig'] + '::C' + (enriched['sig_rank'] % customers_per_sig).astype(str)

# Spend per customer

# customer_sales = enriched.groupby('customer_id', as_index=False).agg(total_sales=('Total','sum')).sort_values('total_sales', ascending=False)

# threshold = customer_sales['total_sales'].quantile(0.90)

# vip_customers = customer_sales[customer_sales['total_sales'] >= threshold]




# enriched['is_vip'] = enriched['customer_id'].isin(vip_customers['customer_id'])




# vip_sales = enriched.loc[enriched['is_vip'], 'Total'].sum()

# nonvip_sales = enriched.loc[~enriched['is_vip'], 'Total'].sum()

# total_sales = enriched['Total'].sum()




# print('VIP customers:', vip_customers.shape[0])

# print('VIP sales:', vip_sales)

# print('Non-VIP sales:', nonvip_sales)

# print('Total sales:', total_sales)

# print(f'VIP sales ratio: {vip_sales / total_sales * 100:.2f}%')


---
## 11. Mini-caso final (tablas para dirección)

Objetivo: construir un paquete mínimo de tablas defendibles.

Preguntas:
1) ¿Qué ciudad y sucursal venden más?
2) ¿Qué línea de producto es más rentable (gross income)?
3) ¿Qué método de pago tiene mayor ticket medio?
4) ¿Qué patrones temporales hay (día/semana/hora)?

Entrega final: un diccionario `report` con 5 tablas.

### Ejercicio 12 (build_report)

Implementa `build_report(df)` que devuelva:
- `sales_by_city_branch`
- `profit_by_product_line`
- `ticket_by_payment`
- `sales_by_weekday`
- `sales_by_hour`

Reglas:
- tablas ordenadas de mayor a menor por la métrica principal
- nombres de columnas claros
- sin índices raros (usa `reset_index()` cuando aplique)


In [29]:
# TODO
def build_report(data: pd.DataFrame) -> dict:
    pass

# report = build_report(df2)
# for k, v in report.items():
#     print('\n===', k, '===')
#     display(v.head(10) if hasattr(v, 'head') else v)

---
## 12. Ejercicios extra (por si sobra tiempo)

1) Construye una tabla Pareto por `Product line`: ordena por ventas y calcula % acumulado.
2) Calcula correlación entre `Rating` y `Total` por ciudad.
3) Encuentra el top 5 de tickets por `gross income` y analiza qué tienen en común.
4) Crea un pivot `weekday × Product line` (suma de ventas) y detecta el mejor día por categoría.


In [30]:
# TODO (extra)
pass

---
## 13. Cierre de Pandas: nombres de columnas, orden, filtros e índice (estilo “producción”)

### Objetivo
Ahora que ya hemos hecho el análisis, dejamos el dataset listo para:
- trabajar más rápido (nombres consistentes)
- evitar errores (columnas sin espacios/símbolos)
- facilitar reporting y visualización
- exportar una versión “clean”

> Nota: lo hacemos al final para no romper el notebook anterior.

In [31]:
df_final = df2.copy()
df_final.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,datetime,month,weekday,hour
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,2019-01-05,13:08,Ewallet,522.83,4.761905,26.1415,9.1,2019-01-05 13:08:00,2019-01,Saturday,13
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,2019-03-08,10:29,Cash,76.40,4.761905,3.8200,9.6,2019-03-08 10:29:00,2019-03,Friday,10
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,2019-03-03,13:23,Credit card,324.31,4.761905,16.2155,7.4,2019-03-03 13:23:00,2019-03,Sunday,13
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,2019-01-27,20:33,Ewallet,465.76,4.761905,23.2880,8.4,2019-01-27 20:33:00,2019-01,Sunday,20
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2019-02-08,10:37,Ewallet,604.17,4.761905,30.2085,5.3,2019-02-08 10:37:00,2019-02,Friday,10


### 13.1 Renombrado manual (cuando quieres control total)

Útil cuando hay pocas columnas “problemáticas” y quieres nombres específicos.

In [32]:
df_final = df_final.rename(columns={
    "Invoice ID": "invoice_id",
    "Customer type": "customer_type",
    "Product line": "product_line",
    "Unit price": "unit_price",
    "Tax 5%": "tax_5",
    "gross margin percentage": "gross_margin_pct",
    "gross income": "gross_income"
})

df_final.columns

Index(['invoice_id', 'Branch', 'City', 'customer_type', 'Gender', 'product_line', 'unit_price', 'Quantity', 'tax_5', 'Total', 'Date', 'Time', 'Payment', 'cogs',
       'gross_margin_pct', 'gross_income', 'Rating', 'datetime', 'month', 'weekday', 'hour'],
      dtype='str')

### 13.2 Renombrado automático (cuando quieres consistencia rápida)

Convierte todo a snake_case:
- espacios → `_`
- minúsculas
- `%` → `pct`

In [33]:
df_auto = df2.copy()

df_auto.columns = (
    df_auto.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
      .str.replace("%", "pct", regex=False)
)

df_auto.columns

Index(['invoice_id', 'branch', 'city', 'customer_type', 'gender', 'product_line', 'unit_price', 'quantity', 'tax_5pct', 'total', 'date', 'time', 'payment', 'cogs',
       'gross_margin_percentage', 'gross_income', 'rating', 'datetime', 'month', 'weekday', 'hour'],
      dtype='str')

### 13.3 Reordenar columnas

En reporting suele interesar primero:
- ids / dimensiones
- tiempo
- métricas

In [34]:
preferred_order = [
    "invoice_id", "branch", "city", "customer_type", "gender",
    "product_line", "payment",
    "date", "time", "datetime", "month", "weekday", "hour",
    "unit_price", "quantity", "tax_5", "total", "cogs",
    "gross_income", "gross_margin_pct", "rating"
]

# keep only columns that exist (robust)
existing = [c for c in preferred_order if c in df_final.columns]
rest = [c for c in df_final.columns if c not in existing]
df_final = df_final[existing + rest]

df_final.head()

,invoice_id,customer_type,product_line,datetime,month,weekday,hour,unit_price,tax_5,cogs,gross_income,gross_margin_pct,Branch,City,Gender,Quantity,Total,Date,Time,Payment,Rating
0,750-67-8428,Member,Health and beauty,2019-01-05 13:08:00,2019-01,Saturday,13,74.69,26.1415,522.83,26.1415,4.761905,A,Yangon,Female,7,548.9715,2019-01-05,13:08,Ewallet,9.1
1,226-31-3081,Normal,Electronic accessories,2019-03-08 10:29:00,2019-03,Friday,10,15.28,3.8200,76.40,3.8200,4.761905,C,Naypyitaw,Female,5,80.2200,2019-03-08,10:29,Cash,9.6
2,631-41-3108,Normal,Home and lifestyle,2019-03-03 13:23:00,2019-03,Sunday,13,46.33,16.2155,324.31,16.2155,4.761905,A,Yangon,Male,7,340.5255,2019-03-03,13:23,Credit card,7.4
3,123-19-1176,Member,Health and beauty,2019-01-27 20:33:00,2019-01,Sunday,20,58.22,23.2880,465.76,23.2880,4.761905,A,Yangon,Male,8,489.0480,2019-01-27,20:33,Ewallet,8.4
4,373-73-7910,Normal,Sports and travel,2019-02-08 10:37:00,2019-02,Friday,10,86.31,30.2085,604.17,30.2085,4.761905,A,Yangon,Male,7,634.3785,2019-02-08,10:37,Ewallet,5.3


### 13.4 Ordenar por múltiples columnas

Ejemplo típico: ver tickets más altos por ciudad.

In [35]:
df_final.sort_values(["city", "total"], ascending=[True, False]).head(10)

KeyError: 'city'

### 13.5 Filtro legible con `query`

`query` suele ser más legible que encadenar condiciones con `&`.

In [ ]:
df_final.query("city == 'Yangon' and total > 300").head()

### 13.6 `value_counts()` para proporciones

Muy útil para distribución de categorías (método de pago, línea de producto, etc.).

In [ ]:
df_final["payment"].value_counts()

In [ ]:
df_final["payment"].value_counts(normalize=True)

### 13.7 Índice: `set_index` / `reset_index`

La mayoría de errores con `groupby`/`pivot` vienen de:
- índices jerárquicos
- no hacer `reset_index()` a tiempo

In [ ]:
by_day = (
    df_final
      .set_index("datetime")["total"]
      .resample("D")
      .sum()
      .reset_index()
      .rename(columns={"datetime": "date", "total": "daily_total"})
)

by_day.head()

### 13.8 Drop de columnas (cuando ya no aportan)

Ejemplo: eliminar columnas redundantes o que no se usarán en visualización.

In [ ]:
df_viz = df_final.drop(columns=["time"], errors="ignore")
df_viz.head()

### 13.9 Exportar versión final (opcional)

Esto simula el paso real de “persistir” un dataset limpio.

In [ ]:
# Uncomment if you want to save locally
# df_viz.to_csv("supermarket_sales_clean.csv", index=False)
# print("Saved: supermarket_sales_clean.csv")

### Ejercicio 13 (cierre)
1) Elige si prefieres renombrado manual o automático y aplícalo.
2) Reordena columnas con un criterio razonable.
3) Crea 2 filtros con `query` (uno por ciudad y otro por product_line).
4) Muestra la distribución de `payment` en %.
5) Genera una tabla diaria de ventas (`resample('D')`) y déjala lista para visualización.

In [ ]:
# TODO
pass